**Configuration note:** the input paths used by this notebook mirror `config.sh` at the repository root. The variables are defined in the next cell — edit them (or `config.sh`) to point at your own data. The remaining cells still contain the original absolute paths from the primary run.


In [ ]:
# Input paths — edit these (or config.sh at the repo root) for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"
CODE_DIR  = f"{PROJ_ROOT}/code/command-line-script"

mito_blast_out = f"{CODE_DIR}/assigning-chromosomes/mitochrondria/hifiasm_041425_mito_blast.out"
coverage_short = f"{CODE_DIR}/contig-coverage/1-allreads-coverage/coverage_short_read_hifiasm_041425_scaffolded_juiceBox_sorted_contig.tsv"
coverage_long  = f"{CODE_DIR}/contig-coverage/coverage_long_read_hifiasm_041425_contig.tsv"
fai            = f"{PROJ_ROOT}/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly/deNovo_hifiHiCMode_hifiData_aggressivePurge3_kmer21_041325.asm.hic.p_ctg.fa.fai"



## Goal of this notebook is to find the contigs from the mitochondrial chromosomes 
Important point to note is that we do not want to filter out the nuclear mitochondrial DNA segments. We found around 755 NUMTs in the human genome. Criteria:
1. Present in the blast table
2. Contains much higher sequencing coverage > 100 fold difference 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import matplotlib.lines as mlines
from Bio import SeqIO


In [ ]:
# Read in the mitochondrial blast table 
mito_table = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/assigning-chromosomes/mitochrondria/hifiasm_041425_mito_blast.out", sep="\t",names=['qseqid','sseqid','pident','qcovs','length','gapopen','evalue','bitscore','mismatch'])
print("Mito Table:")
display(mito_table.head())
print(mito_table.shape)

# Read in the assembly table
assembly_table = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly/deNovo_hifiHiCMode_hifiData_aggressivePurge3_kmer21_041325.asm.hic.p_ctg.fa.fai", sep="\t", names=['chrom','length','x','x1','x2']).iloc[:, :2]
print("\nAssembly Table:")
display(assembly_table.head())

# Read in the long_reads coverage table
coverage_table = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/contig-coverage/coverage_long_read_hifiasm_041425_contig.tsv", sep="\t", names=['contig','start_pos','end_pos','coverage'])
print("\nCoverage Table:")
display(coverage_table.head())

# # Read in the short_reads coverage table
# coverage_table = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/contig-coverage/1-allreads-coverage/coverage_short_read_hifiasm_041425_scaffolded_juiceBox_sorted_contig.tsv", sep="\t", names=['contig','start_pos','end_pos','coverage'])
# print("\nCoverage Table:")
# display(coverage_table.head())




In [ ]:
print("How many different contigs have been identified") 

mito_contigs = mito_table["sseqid"].unique()
len(mito_contigs)

In [ ]:
print("What is the length distribution of the mitochondrial contigs")
plot_mito_length = assembly_table[assembly_table["chrom"].isin(mito_contigs)]
plot_mito_length
plt.hist(plot_mito_length["length"],density=False, bins=100)

In [ ]:
## Theoretical mitochondrial contigs because the mito chrom is about 16 kb
plot_mito_length[plot_mito_length["length"]<20000]

In [ ]:
plot_mito_length[plot_mito_length["length"]<20000]

In [ ]:
plot_mito_length[plot_mito_length["length"]<20000][["length"]]

In [ ]:
plot_mito_length[plot_mito_length["length"]<16798][["length"]]


In [ ]:
plt.hist(plot_mito_length[plot_mito_length["length"]<20000][["length"]]) # 16kb is the size of the mitochondrial chromosome size

In [ ]:
plt.hist(plot_mito_length[plot_mito_length["length"]<200000][["length"]], bins=11) # 16kb is the size of the mitochondrial chromosome size

In [ ]:
## Waiting for the coverage analysis to be done 
plt.hist(coverage_table["coverage"])

In [ ]:
coverage_table.head()

In [ ]:
alignment_plot_table = pd.merge(mito_table, assembly_table, left_on="sseqid", right_on="chrom", how="left")
alignment_plot_table = pd.merge(alignment_plot_table, coverage_table, left_on="sseqid", right_on="contig", how="left")
alignment_plot_table=alignment_plot_table.rename(columns={
    'length_x': 'alignment_length',
    'length_y': 'contig_length'
})
alignment_plot_table

In [ ]:
print(alignment_plot_table.shape)
alignment_plot_table.head()

In [ ]:
## Do a scatter plot of coverage and length and circle 
plt.scatter(alignment_plot_table["alignment_length"], alignment_plot_table["coverage"])
plt.axvline(x=16798, color='r', linestyle='--', linewidth=1.5)

# Add labels and title
plt.xlabel("Length of mitochondrial alignment")
plt.ylabel("Contig Coverage")
plt.title("Mitochondrial chrom alignment vs coverage")
## None of the alignment could be larger than mito chrom size anyways 

In [ ]:
# Create collapsed dataframe (if not already done)
contig_df = alignment_plot_table.groupby('contig').agg({
    'alignment_length': 'sum',  # Total aligned length
    'pident': 'mean',
    'contig_length': 'first',  # Contig length (same for all rows)
    'coverage': 'first',  # Coverage (same for all rows)
    'chrom': 'first'
})
contig_df['alignment_percent'] = contig_df['alignment_length']/contig_df['contig_length'] * 100
contig_df

In [ ]:
# Classification and visualization (with marker shapes)
threshold = 100000
contig_df['group'] = np.where(
    contig_df['contig_length'] >= threshold,
    'Contigs with NUMTs',
    "Mitochondrial contig candidates"
)

# Get the corresponding #
num_numts = len(contig_df[contig_df['group'] == "Contigs with NUMTs"])
num_mito = len(contig_df[contig_df['group'] =="Mitochondrial contig candidates"])

# Set global font sizes
title_fontsize = 27
axis_label_fontsize = 27
tick_fontsize = 23
legend_fontsize = 23
colorbar_fontsize = 23

# Custom legend handles (black markers with the correct shape)
legend_numts = mlines.Line2D([], [], color='black', marker='s', linestyle='None',
                             markersize=8, label=f"NUMTs contigs candidates (n={num_numts})")
legend_mito = mlines.Line2D([], [], color='black', marker='o', linestyle='None',
                             markersize=8, label= f"Mitochondrial contig candidates (n={num_mito})")
legend_line = mlines.Line2D([], [], color='red', linestyle='--',
                            label='Reference mitochondrial length')

# Split data
above = contig_df[contig_df['group'] == 'Contigs with NUMTs']
below = contig_df[contig_df['group'] == "Mitochondrial contig candidates"]

# Get global color limits
vmin = contig_df['alignment_percent'].min()
vmax = contig_df['alignment_percent'].max()

plt.figure(figsize=(15, 5))

# Create scatter plots with different markers
sc1 = plt.scatter(
    x=np.log10(above['contig_length']),
    y=above['coverage'],
    c=above['alignment_percent'],
    s=50,
    marker='s',  # Square for NUMTs
    cmap='bwr',
    alpha=0.7,
    edgecolor='w',
    vmin=vmin,
    vmax=vmax,
)

sc2 = plt.scatter(
    x=np.log10(below['contig_length']),
    y=below['coverage'],
    c=below['alignment_percent'],
    s=50,
    marker='o',  # Circle for candidates
    cmap='bwr',
    alpha=0.7,
    edgecolor='w',
    vmin=vmin,
    vmax=vmax,
)

# Add reference line (Degu mitochondrial length)
plt.axvline(
    x=np.log10(16798),
    color='r',
    linestyle='--')

cbar = plt.colorbar(sc2)
cbar.set_label('Percentage Aligned (%)', fontsize=colorbar_fontsize)
cbar.ax.tick_params(labelsize=tick_fontsize)  # Colorbar tick labels

# Formatting
plt.xlabel('Log10(Contig Length)', fontsize=axis_label_fontsize)
plt.ylabel('Coverage of Hifi reads', fontsize=axis_label_fontsize)
plt.title('Identifying mitochondrial contigs', fontsize=title_fontsize)

# Set tick label sizes
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# Legend with custom font size
legend = plt.legend(handles=[legend_numts, legend_mito, legend_line], 
                   fontsize=legend_fontsize,
                   frameon=True,
                   fancybox=True,
                   shadow=True)

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Classification and visualization (with marker shapes)
threshold = 100000
contig_df['group'] = np.where(
    contig_df['contig_length'] >= threshold,
    'Contigs with NUMTs',
    "Mitochondrial contig candidates"
)

# Get the corresponding #
num_numts = len(contig_df[contig_df['group'] == "Contigs with NUMTs"])
num_mito = len(contig_df[contig_df['group'] =="Mitochondrial contig candidates"])

# Custom legend handles (black markers with the correct shape)
legend_numts = mlines.Line2D([], [], color='black', marker='s', linestyle='None',
                             markersize=8, label=f"NUMTs contigs candidates (n={num_numts})")
legend_mito = mlines.Line2D([], [], color='black', marker='o', linestyle='None',
                             markersize=8, label= f"Mitochondrial contig candidates (n={num_mito})")
legend_line = mlines.Line2D([], [], color='red', linestyle='--',
                            label='Reference mitochondrial length')

# Split data
above = contig_df[contig_df['group'] == 'Contigs with NUMTs']
below = contig_df[contig_df['group'] == "Mitochondrial contig candidates"
]
# Get global color limits
vmin = contig_df['alignment_percent'].min()
vmax = contig_df['alignment_percent'].max()

plt.figure(figsize=(18, 5))

# Create scatter plots with different markers
sc1 = plt.scatter(
    x=np.log10(above['contig_length']),
    y=above['coverage'],
    c=above['alignment_percent'],
    s=50,
    marker='s',  # Square for NUMTs
    cmap='viridis',
    alpha=0.7,
    edgecolor='w',
    vmin=vmin,  # Critical fix
    vmax=vmax,  # Critical fix
)

sc2 = plt.scatter(
    x=np.log10(below['contig_length']),
    y=below['coverage'],
    c=below['alignment_percent'],
    s=50,
    marker='o',  # Circle for candidates
    cmap='viridis',
    alpha=0.7,
    edgecolor='w',
    vmin=vmin,  # Critical fix
    vmax=vmax,  # Critical fix
)

# Add reference line (Degu mitochondrial length)
plt.axvline(
    x=np.log10(16798),
    color='r',
    linestyle='--')

cbar = plt.colorbar(sc2)
cbar.set_label('Percentage Aligned (%)')

# Formatting
plt.xlabel('Log10(Contig Length)',size=27)
plt.ylabel('Coverage of Hifi reads',size=27)
plt.title('Contigs wtih mitochondrial alignment',size=27)
plt.legend(handles=[legend_numts, legend_mito, legend_line])
# plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
print(below.shape)
print(above.shape)

In [ ]:
below.to_csv("mito_contigs.txt", sep="\t")

In [ ]:
## Merge the information back to the alignments 
alignment_plot_table_final = alignment_plot_table.merge( contig_df["group"], left_on="sseqid", right_index=True, how="left")
alignment_plot_table_final

In [ ]:
# Get the corresponding alignements?
num_numts = len(alignment_plot_table_final[alignment_plot_table_final['group'] == "Contigs with NUMTs"])
num_mito = len(alignment_plot_table_final[alignment_plot_table_final['group'] =="Mitochondrial contig candidates"])

# Custom legend handles (black markers with the correct shape)
legend_numts = mlines.Line2D([], [], color='black', marker='s', linestyle='None',
                             markersize=8, label=f'Likely NUMTs alignments (n={num_numts})')
legend_mito = mlines.Line2D([], [], color='black', marker='o', linestyle='None',
                             markersize=8, label=f"Likely mitochondrial alignments (n={num_mito})")
legend_line = mlines.Line2D([], [], color='red', linestyle='--',
                            label='Reference mitochondrial length')

# # Split data
above = alignment_plot_table_final[alignment_plot_table_final['group'] == 'Contigs with NUMTs']
below = alignment_plot_table_final[alignment_plot_table_final['group'] == 'Mitochondrial contig candidates']
# Get global color limits
vmin = alignment_plot_table_final['pident'].min()
vmax = alignment_plot_table_final['pident'].max()

plt.figure(figsize=(10, 6))

# Create scatter plots with different markers
sc1 = plt.scatter(
    x=above['alignment_length'],
    y=above['coverage'],
    c=above['pident'],
    s=50,
    marker='s',  # Square for NUMTs
    cmap='viridis',
    alpha=0.7,
    edgecolor='w',
    vmin=vmin,  # Critical fix
    vmax=vmax,  # Critical fix
)

sc2 = plt.scatter(
    x=below['alignment_length'],
    y=below['coverage'],
    c=below['pident'],
    s=50,
    marker='o',  # Circle for candidates
    cmap='viridis',
    alpha=0.7,
    edgecolor='w',
    vmin=vmin,  # Critical fix
    vmax=vmax,  # Critical fix
)

# Add reference line (Degu mitochondrial length)
plt.axvline(
    x=16798,
    color='r',
    linestyle='--')

cbar = plt.colorbar(sc2)
cbar.set_label('Percent Aligned identity(%)')

# # Formatting
plt.xlabel('Alignment Length')
plt.ylabel('Coverage of Hifi reads')
plt.title('Mitochondrial alignment analysis')
plt.legend(handles=[legend_numts, legend_mito, legend_line])
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
with open("mito_contigs_041425.txt") as f:
    contigs_to_remove = set(line.split()[0] for line in f)

contigs_to_remove=list(contigs_to_remove)
exclude=contigs_to_remove

In [ ]:
# Process FASTA and write output
with open('/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/0414250-assembly/hifiasm-041425-assembly/deNovo_hifiHiCMode_hifiData_aggressivePurge3_kmer21_041325.asm.hic.p_ctg.fa') as fin, open('/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/0414250-assembly/hifiasm-041425-assembly-mitoFiltered/genome_chrom.fasta', 'w') as fout:
    write_contig = False
    for line in fin:
        if line.startswith('>'):
            # Extract contig name from header (after '>' and before first space)
            contig_name = line[1:].split(maxsplit=1)[0].strip()
            write_contig = contig_name not in exclude
        if write_contig:
            fout.write(line)

In [ ]:
contigs_to_remove